In [53]:
# # Description
# 
# Prove that the Noise Reduction model works on a 2D polinomial.

# Add libs folder path to system variables to make the importation of local libraries easier.
# __________________________________________________________________________________________________
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import TensorDataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [54]:
def add_gaussian_noise(df:pd.DataFrame, columns:List[str], mean:float = 0.0, std:float = 0.1):
    """
    Adds Gaussian noise to specified columns in a DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame to which noise will be added.
        columns (list): A list of column names in the DataFrame to which noise will be added.
        mean (int, optional): The mean of the Gaussian distribution. Defaults to 0.
        std (float, optional): The standard deviation of the Gaussian distribution. Defaults to 0.1.

    Returns:
        pandas.DataFrame: The DataFrame with added Gaussian noise to specified columns.
    """
    for col in columns:
        df[col] = df[col] + np.random.normal(loc=mean, scale=std, size=len(df))
    return df

In [55]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets

X = X.drop(columns=['Sex'], inplace=False)

df_data = pd.DataFrame(X, columns=abalone.data.feature_names)
df_data['target'] = y

scaler = MinMaxScaler()
df_data = pd.DataFrame(scaler.fit_transform(df_data), columns=df_data.columns)

X = df_data.drop(columns=['target'], inplace=False)
y = df_data['target']

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# No noise XAI Benchmarck

In [57]:
model_params = {
    'ridge': {"alpha": 1.0},
    'pls': {"n_components": 1},
    'tree': {"max_depth": 5},
    'svm': {"dual": 'auto'},
    'knn': {"n_neighbors": 5, "weights": 'uniform', "algorithm": 'auto', "leaf_size": 30, "p": 2, "n_jobs": None},
}
xai_benchmark = XAI_benchmark(is_ts = False, model_params = model_params)

In [58]:
xai_benchmark.fit(X_train, y_train.values.ravel())

Fitting Ridge model...
Fitting Partial Least Squares model...
Fitting Decision Tree model...
Fitting Support Vector Machine model...
Fitting K-Nearest Neighbours model...
All models fitted!


/mnt/homeGPU/JJavierAR/repsol_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [59]:
xai_benchmark.predict(X_test, y_test.values.ravel(), get_metrics=True)

({'ridge': array([ 0.38003037,  0.3275612 ,  0.43924189,  0.39214949,  0.34038061,
          0.33834242,  0.31818762,  0.27100008,  0.22968997,  0.34825369,
          0.38118863,  0.23758119,  0.37401718,  0.24148964,  0.27414924,
          0.40956867,  0.40843408,  0.33975017,  0.47395012,  0.26962057,
          0.64823593,  0.26758322,  0.36580726,  0.37264421,  0.30940389,
          0.25590614,  0.32409632,  0.26229435,  0.32014026,  0.30898338,
          0.33824979,  0.37167707,  0.22961759,  0.31457676,  0.29328786,
          0.29407342,  0.20137304,  0.12366368,  0.25910555,  0.21811942,
          0.39917119,  0.5677922 ,  0.40025956,  0.30591948,  0.35541355,
          0.38227802,  0.30652937,  0.29525298,  0.38921719,  0.15909516,
          0.21778531,  0.60732457,  0.32468314,  0.14615722,  0.21936232,
          0.34905272,  0.35931662,  0.45979775,  0.3367447 ,  0.22910169,
          0.38059801,  0.44458516,  0.3436406 ,  0.24923187,  0.25361468,
          0.35727369,  0.1623

In [61]:
type(y), y.shape

(pandas.core.series.Series, (4177,))

In [62]:
# Transform the data into a tensor
train_dataset = TensorDataset(
    x=X_train.values,
    y=y_train.values.reshape(-1, 1)
)
# Transform the data into a tensor
val_dataset = TensorDataset(
    x=X_test.values,
    y=y_test.values.reshape(-1, 1)
)

# Create the dataloaders
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [63]:
model = GridFullyDenseNN(
    n_layers=4,
    hidden_layers=[
        (X_train.shape[1], 1024),
        (1024, 512),
        (512, 1024),
        (1024, 1),
    ],
    dropout_layers=[
        0.0,
        0.0,
        0.0,
        0.0,
    ],
    activation_func_layers=[
        nn.Sigmoid(),
        nn.Sigmoid(),
        nn.Sigmoid(),
        nn.Sigmoid(),
    ],
    want_dropout=[
        False,
        False,
        False,
        False,
    ],
    want_linear=[
        True,
        True,
        True,
        True,
    ],
    want_activation=[
        True,
        True,
        True,
        True,
    ],
)
model.to(device)

GridFullyDenseNN(
  (model): Sequential(
    (linear_0): Linear(in_features=7, out_features=1024, bias=True)
    (activation_0): Sigmoid()
    (linear_1): Linear(in_features=1024, out_features=512, bias=True)
    (activation_1): Sigmoid()
    (linear_2): Linear(in_features=512, out_features=1024, bias=True)
    (activation_2): Sigmoid()
    (linear_3): Linear(in_features=1024, out_features=1, bias=True)
    (activation_3): Sigmoid()
  )
)

In [64]:
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=model,
    train_generator=train_dataloader,
    val_generator=val_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=os.path.join(CHECKPOINT_PATH, 'abalone_gradient')
)

Train the model

In [66]:
type(y), y.shape

(pandas.core.series.Series, (4177,))

In [67]:
model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)

Training is started!


Epochs loop:  22%|▏| 109/500 [01:31<05:26,  1.20epoch/s, Train loss: 0.006245847762163

Training stopped as validation loss did not improve for 15 epochs.


or load a checkpoint

In [ ]:
# model.load_state_dict(torch.load(os.path.join(CHECKPOINT_PATH, f'model_noise_{sigma}_polinomial.pth')))

In [69]:
y_pred_test = model(torch.tensor(X_test.values).float().to(device)).cpu().detach().numpy().reshape(-1)

In [70]:
# Show the metrics
gt_values =y_test.values
predicted_values = y_pred_test

mae = mean_absolute_error(gt_values, predicted_values)
mse = mean_squared_error(gt_values, predicted_values)
rmse = np.sqrt(mse)
# mape = np.mean(np.abs(gt_values - predicted_values) / gt_values) * 100
r_squared = r2_score(gt_values, predicted_values)
nn_benchmark = {
    'nn': {
        "MSE: ": mse,
        "RMSE: ": rmse,
        "MAE: ": mae,
        # "MAPE: ": mape,
        "R2: ": r_squared
    }
}
nn_benchmark

{'nn': {'MSE: ': 0.0062087007127492825,
  'RMSE: ': 0.07879530895141716,
  'MAE: ': 0.057160222076607835,
  'R2: ': 0.5503439343615317}}

# Gradients to reduce noise

In [72]:
X_no_noise = X.copy()
y_no_noise = y.copy()

In [74]:
dlnr = DLNoiseReduction(model=model, criterion=criterion)
dlnr.fit(X.values, y.values.reshape(-1, 1))

In [75]:
X_no_noise, y_no_noise = dlnr.transform(
                                        nrr=0.05,
                                        nr_threshold=0.01,
                                        max_epochs=200,
                                        plot_progress=False,
                                        path_to_save_imgs=None
)

# Denoised XAI Benchmark

In [80]:
X_no_noise.shape, y_no_noise.shape

((4177, 7), (4177, 1))

In [82]:
xai_benchmark.fit(X_no_noise, y_no_noise.ravel())

Fitting Ridge model...
Fitting Partial Least Squares model...
Fitting Decision Tree model...
Fitting Support Vector Machine model...
Fitting K-Nearest Neighbours model...
All models fitted!


/mnt/homeGPU/JJavierAR/repsol_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [83]:
xai_benchmark.predict(X_no_noise, y_no_noise.ravel(), get_metrics=True)

({'ridge': array([0.3808067 , 0.22560773, 0.31732679, ..., 0.32637955, 0.32537141,
         0.36094089]),
  'pls': array([0.2854467 , 0.22850915, 0.30537205, ..., 0.37772531, 0.36455042,
         0.45262714]),
  'decision_tree': array([0.43224482, 0.2243419 , 0.30329068, ..., 0.36365667, 0.33755796,
         0.41442893]),
  'svm': array([0.38883678, 0.22106584, 0.31075083, ..., 0.32939567, 0.31372813,
         0.36012459]),
  'knn': array([0.42422281, 0.22010753, 0.31111803, ..., 0.33071849, 0.32185536,
         0.36795514])},
 {'ridge': {'mse': 0.0004957835672751695,
   'rmse': 0.022266197863020296,
   'mae': 0.016485324725790715,
   'R2': 0.9472678499714333},
  'pls': {'mse': 0.004591704147399826,
   'rmse': 0.0677621143958763,
   'mae': 0.0506648402954385,
   'R2': 0.5116206990920849},
  'decision_tree': {'mse': 0.0011925222073053124,
   'rmse': 0.034532914839400866,
   'mae': 0.025098782997582572,
   'R2': 0.8731618712301547},
  'svm': {'mse': 0.0004910520884189694,
   'rmse': 0.02

In [85]:
xai_benchmark.predict(X_test.values, y_test.values, get_metrics=True)

({'ridge': array([ 0.38801329,  0.3340337 ,  0.45649056,  0.39803055,  0.34890248,
          0.3518775 ,  0.3273965 ,  0.26806483,  0.23136642,  0.35036664,
          0.39262491,  0.24282356,  0.37763897,  0.24248893,  0.27561107,
          0.42191817,  0.41763484,  0.34313655,  0.48419117,  0.27115021,
          0.66657261,  0.26203744,  0.36855884,  0.37845535,  0.31079558,
          0.25823758,  0.3313375 ,  0.26211663,  0.32659554,  0.31533923,
          0.33677495,  0.37579986,  0.23174747,  0.32387725,  0.30084295,
          0.28874448,  0.20086046,  0.1126833 ,  0.26019966,  0.21605625,
          0.39066954,  0.5865005 ,  0.40264225,  0.31158072,  0.35550782,
          0.37094624,  0.30506223,  0.2952102 ,  0.39307508,  0.15976588,
          0.21756869,  0.61854079,  0.33008029,  0.14067482,  0.22038559,
          0.35415069,  0.36494169,  0.46734666,  0.34421555,  0.23195666,
          0.38170894,  0.45729686,  0.34700186,  0.25027106,  0.25032835,
          0.35881622,  0.1577